# C17 — Data Scrubbing + GroupDRO + sqrt_rw

**Tier-3 combo — fairness-first.**

Scrub city/age tokens + GroupDRO (exponential group reweighting by city) + sqrt city sample weights, 2 epochs.

**Гипотеза:** GroupDRO тянет worst gap вниз (single G → 0.276), scrubbing даёт flip ≈ 0 как в C11/C13.

**Runs**
- `scrubbing_gdro_eta005_2ep` — η=0.05 (мягче, выше F1)
- `scrubbing_gdro_eta01_2ep` — η=0.10 (агрессивнее gap)

**Reference:** G η=0.1 alone → gap 0.276, F1 0.562; C13 Scrub+R-Drop → gap 0.324, flip ≈ 0.

**После обучения:** city-swap eval → `c18_scrubbing_groupdro_city_swap_eval.ipynb`.

**Outputs**
- Models: `notebooks/models/challengers/scrubbing_gdro_eta005_2ep/`, `..._eta01_2ep/`
- Summary: `figures/challengers/c17_scrubbing_groupdro_summary.csv`
- Results: `notebooks/results/challenger_training/c17_scrubbing_groupdro/`

In [1]:
import json
import random
import re

import joblib
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = NOTEBOOKS_DIR / "models" / "challengers"
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
RESULTS_DIR = NOTEBOOKS_DIR / "results" / "challenger_training" / "c17_scrubbing_groupdro"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = FIGURES_DIR / "c17_scrubbing_groupdro_summary.csv"

RUNS = [
    {"tag": "eta005_2ep", "eta": 0.05, "epochs": 2},
    {"tag": "eta01_2ep", "eta": 0.10, "epochs": 2},
]

print(f"CWD: {CWD}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR exists: {(DATA_DIR / 'train.csv').exists()}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"Runs: {[r['tag'] for r in RUNS]}")

CWD: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/challengers
PROJECT_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
DATA_DIR exists: True
MODELS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers
FIGURES_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers
RESULTS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/challenger_training/c17_scrubbing_groupdro
Runs: ['eta005_2ep', 'eta01_2ep']


In [3]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_val = pd.read_csv(DATA_DIR / "val.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

mapping = pd.read_csv(DATA_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)

city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)

city_to_id = {c: i for i, c in enumerate(df_train["city_group"].unique())}
num_groups = len(city_to_id)
df_train["city_id"] = df_train["city_group"].map(city_to_id).astype(int)
df_val["city_id"] = df_val["city_group"].map(city_to_id).fillna(-1).astype(int)
df_test["city_id"] = df_test["city_group"].map(city_to_id).fillna(-1).astype(int)

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"Labels: {num_labels}, City groups: {num_groups}")

Train: 16530, Val: 5510, Test: 5510
Labels: 9, City groups: 41


In [4]:
CITY_WORDS = [
    "москва", "московская", "московский", "мск",
    "санкт-петербург", "петербург", "спб", "питер", "ленинград",
    "новосибирск", "екатеринбург", "казань", "нижний новгород",
    "челябинск", "самара", "омск", "ростов-на-дону", "уфа",
    "красноярск", "воронеж", "пермь", "волгоград",
    "краснодар", "саратов", "тюмень", "тольятти", "ижевск",
    "барнаул", "ульяновск", "иркутск", "хабаровск", "ярославль",
    "владивосток", "махачкала", "томск", "оренбург", "кемерово",
    "новокузнецк", "рязань", "астрахань", "пенза", "липецк",
    "калининград", "тула", "курск", "ставрополь", "сочи",
    "минск", "алматы", "киев", "симферополь",
    "область", "край", "республика", "регион",
    "забайкальский", "приморский", "краснодарский",
]

AGE_WORDS = [
    "пенсионер", "пенсионерка", "пенсия", "пенсионный",
    "студент", "студентка", "выпускник", "выпускница",
    "молодой", "молодая", "junior", "senior",
]

ALL_SENSITIVE = set(CITY_WORDS + AGE_WORDS)


def scrub_text(text, mask_token="[MASK]"):
    if pd.isna(text):
        return ""
    result = str(text)
    for word in sorted(ALL_SENSITIVE, key=len, reverse=True):
        pattern = re.compile(re.escape(word), re.IGNORECASE)
        result = pattern.sub(mask_token, result)
    return result


print("Scrubbing texts...")
for df in [df_train, df_val, df_test]:
    df["resume_text_scrubbed"] = df["resume_text"].apply(scrub_text)

modified = (df_train["resume_text"] != df_train["resume_text_scrubbed"]).sum()
print(f"Sensitive words: {len(ALL_SENSITIVE)}")
print(f"Modified {modified}/{len(df_train)} training texts ({100 * modified / len(df_train):.1f}%)")

Scrubbing texts...
Sensitive words: 70
Modified 14428/16530 training texts (87.3%)


In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize(batch):
    return tokenizer(
        batch["resume_text_scrubbed"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )


train_ds = Dataset.from_pandas(
    df_train[["resume_text_scrubbed", "y", "city_id", "sample_weight"]]
).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(
    df_val[["resume_text_scrubbed", "y", "city_id"]]
).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(
    df_test[["resume_text_scrubbed", "y", "city_id"]]
).map(tokenize, batched=True)

train_ds = train_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id", "sample_weight"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id"])

Map: 100%|██████████| 5510/5510 [00:04<00:00, 1279.32 examples/s]


In [ ]:
class GroupDROTrainer(Trainer):
    def __init__(self, *args, num_groups, eta=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_groups = num_groups
        self.eta = eta
        self.q = torch.ones(num_groups) / num_groups
        print(f"GroupDRO: {num_groups} groups, eta={eta}")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        city_id = inputs.pop("city_id")
        sample_weight = inputs.pop("sample_weight", None)
        outputs = model(**inputs)

        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        per_sample_loss = loss_fct(outputs.logits, inputs["labels"])
        if sample_weight is not None:
            per_sample_loss = per_sample_loss * sample_weight.to(per_sample_loss.dtype)

        device = per_sample_loss.device
        group_loss = torch.zeros(self.num_groups, device=device)
        group_count = torch.zeros(self.num_groups, device=device)
        for g in range(self.num_groups):
            mask = city_id == g
            if mask.any():
                group_loss[g] = per_sample_loss[mask].mean()
                group_count[g] = mask.sum().float()

        with torch.no_grad():
            q = self.q.to(device) * torch.exp(self.eta * group_loss * (group_count > 0).float())
            self.q = (q / q.sum()).cpu()

        loss = (self.q.to(device) * group_loss).sum()
        return (loss, outputs) if return_outputs else loss


def compute_metrics(prediction_output):
    preds = np.argmax(prediction_output.predictions, axis=1)
    return {
        "accuracy": accuracy_score(prediction_output.label_ids, preds),
        "macro_f1": f1_score(prediction_output.label_ids, preds, average="macro"),
    }


def ovr_rates(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    for gi, group_name in enumerate(groups):
        dg = df[df[group_col] == group_name]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(num_classes):
            positive_mask = yt == c
            tp = np.sum((yp == c) & positive_mask)
            fn = np.sum((yp != c) & positive_mask)
            support[gi, c] = positive_mask.sum()
            tpr[gi, c] = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return tpr, support


def robust_gaps(tpr, support, min_support=30):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[support[:, c] >= min_support, c]
        col = col[~np.isnan(col)]
        gaps.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return (valid.max() if len(valid) else np.nan, valid.mean() if len(valid) else np.nan)


def find_latest_checkpoint(ckpt_dir: Path):
    if not ckpt_dir.exists():
        return None
    ckpts = sorted(
        ckpt_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.rsplit("-", 1)[-1]),
    )
    return str(ckpts[-1]) if ckpts else None

In [7]:
summary_rows = []

for run in RUNS:
    model_name = f"scrubbing_gdro_{run['tag']}"
    save_dir = MODELS_DIR / model_name
    checkpoint_dir = save_dir / "checkpoints"
    run_results_dir = RESULTS_DIR / model_name
    run_results_dir.mkdir(parents=True, exist_ok=True)

    resume_from = find_latest_checkpoint(checkpoint_dir)

    print("\n" + "=" * 80)
    print(f"Training {model_name}: scrubbing + GroupDRO eta={run['eta']}, epochs={run['epochs']}")
    if resume_from:
        print(f"Resuming from checkpoint: {resume_from}")
    else:
        print("No checkpoint found — training from scratch")

    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
    args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=run["epochs"],
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    )

    trainer = GroupDROTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        num_groups=num_groups,
        eta=run["eta"],
    )

    trainer.train(resume_from_checkpoint=resume_from)
    pred_output = trainer.predict(test_ds)
    y_true = pred_output.label_ids
    y_pred = np.argmax(pred_output.predictions, axis=1)

    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

    df_eval = df_test.copy()
    df_eval["y_true"] = y_true
    df_eval["y_pred"] = y_pred
    tpr, support = ovr_rates(df_eval, "city_group", num_labels)
    worst_gap, macro_gap = robust_gaps(tpr, support, min_support=30)

    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(save_dir, safe_serialization=True)
    tokenizer.save_pretrained(save_dir)
    joblib.dump(le, save_dir / "label_encoder.joblib")

    training_config = {
        "method": "Data Scrubbing + GroupDRO + sqrt_rw",
        "model_name": model_name,
        "scrubbing": True,
        "scrub_at_inference": True,
        "scrubbed_words_count": len(ALL_SENSITIVE),
        "groupdro_eta": run["eta"],
        "num_groups": num_groups,
        "epochs": run["epochs"],
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "tpr_gap_worst_robust": float(worst_gap),
        "tpr_gap_macro_robust": float(macro_gap),
    }
    with open(save_dir / "training_config.json", "w", encoding="utf-8") as f:
        json.dump(training_config, f, indent=2, ensure_ascii=False)
    with open(run_results_dir / "training_config.json", "w", encoding="utf-8") as f:
        json.dump(training_config, f, indent=2, ensure_ascii=False)

    pred_df = df_eval[["city_group", "label", "supercategory", "y_true", "y_pred"]].copy()
    pred_df.to_csv(run_results_dir / "predictions_test.csv", index=False)

    summary_rows.append({
        "model_name": model_name,
        "method": "Data Scrubbing + GroupDRO + sqrt_rw",
        "eta": run["eta"],
        "epochs": run["epochs"],
        "accuracy": acc,
        "macro_f1": macro_f1,
        "worst_gap": float(worst_gap),
        "macro_gap": float(macro_gap),
        "model_dir": str(save_dir.relative_to(PROJECT_ROOT)),
    })

    print(f"TEST: Acc={acc:.4f}, Macro-F1={macro_f1:.4f}")
    print(f"FAIRNESS (robust): worst={worst_gap:.4f}, macro={macro_gap:.4f}")
    print(f"Model saved to: {save_dir}")

summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(["worst_gap", "macro_f1"], ascending=[True, False])
    .reset_index(drop=True)
)
summary_df.to_csv(SUMMARY_CSV, index=False)
summary_df.to_csv(RESULTS_DIR / "training_summary.csv", index=False)

print(f"\nSummary saved to: {SUMMARY_CSV}")
summary_df


Training scrubbing_gdro_eta005_2ep: scrubbing + GroupDRO eta=0.05, epochs=2
No checkpoint found — training from scratch


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2332.87it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

GroupDRO: 41 groups, eta=0.05


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.197659,1.083886,0.588022,0.607567
2,0.201703,1.043689,0.594737,0.610482


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.231388,1.147162,0.539564,0.553587
2,0.200424,1.094468,0.582940,0.597187


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.61it/s]

TEST: Acc=0.5768, Macro-F1=0.5885
FAIRNESS (robust): worst=0.3235, macro=0.1268
Model saved to: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/models/challengers/scrubbing_gdro_eta01_2ep

Summary saved to: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers/c17_scrubbing_groupdro_summary.csv


,model_name,method,eta,epochs,accuracy,macro_f1,worst_gap,macro_gap,model_dir
0,scrubbing_gdro_eta01_2ep,Data Scrubbing + GroupDRO + sqrt_rw,0.10,2,0.576770,0.58845,0.323529,0.126763,notebooks/models/challengers/scrubbing_gdro_et...
1,scrubbing_gdro_eta005_2ep,Data Scrubbing + GroupDRO + sqrt_rw,0.05,2,0.588022,0.60063,0.358824,0.117491,notebooks/models/challengers/scrubbing_gdro_et...
